$$
\newcommand{\bolde}{\boldsymbol{e}}
\newcommand{\boldh}{\boldsymbol{h}}
\newcommand{\boldp}{\boldsymbol{p}}
\newcommand{\boldr}{\boldsymbol{r}}
\newcommand{\boldt}{\boldsymbol{t}}
\newcommand{\boldq}{\boldsymbol{q}}
\newcommand{\boldM}{\boldsymbol{M}}
\newcommand{\boldP}{\boldsymbol{P}}
\newcommand{\boldR}{\boldsymbol{R}}
\newcommand{\boldT}{\boldsymbol{T}}
\newcommand{\boldQ}{\boldsymbol{Q}}
\newcommand{\RR}{\mathbb{R}}
$$

<a id='hw_lidar_odometry'></a>
# Домашняя работа. Лидарная одометрия<sup>[toc](#_toc)</sup>

<a id='_toc'></a>
# Содержание<sup>[toc](#_toc)</sup>
1. [Задание](#task)
2. [Варианты лидарной одометрии](#lo_versions)
    * [Лидарная одометрия v1.0](#lo_v1)
    * [Лидарная одометрия v2.0](#lo_v2)
    * [Лидарная одометрия v3.0](#lo_v3)
    * [Лидарная одометрия v4.0](#lo_v4)
3. [KITTI](#kitti)
4. [Реализация лидарной одометрии](#lo_impl)
    * [Подготовительные действия](#lo_impl_prep)
    * [Основная работа](#lo_impl_main)
    * [Анализ результатов](#lo_vis)

<a id='task'></a>
# Задание<sup>[toc](#_toc)</sup>

**Лидарная одометрия** &mdash; это способ оценки перемещения лидара (машины), при котором очередное снятое лидарное облако сопоставляется с предыдущим, чтобы найти относительное перемещение лидара в пространстве. Если говорить более точно, то мы оцениваем, насколько лидар сместился относительно своего предыдущего положения, и через последовательность относительных перемещений лидара восстанавливаем его траекторию в пространстве.

В данной домашней работе предлагается реализовать лидарную одометрию на данных из датасета KITTI. С её помощью требуется определить траекторию движения центрального лидара в пространстве, а также восстановить лидарную карту местности, смержив все лидарные облака воедино.

## Что надо отправить на страницу задания в личном кабинете?
Нужно отправить в одном архиве этот ноутбук + директорию `hw_results` (про неё будет написано далее в ноутбуке). Таким убразом у проверяющих будет возможность провалидировать результаты.
1. В директории `hw_results/2011_09_26-0005/trajectories` должны быть все полученные с помощью лидарной одометрии траектории машины. Их будет порядка 20 (см. пункт про запуск лидарной одометрии с разными параметрами).
2. В директории `hw_results/2011_09_26-0005/visualizations` должно быть две визуализации: одна для `BASELINE`-алгоритма, а вторая для `LIDAR_ODOMETRY`-алгоритма с лучшими найденными параметрами лидарной одометрии.
3. Также в директории `hw_results/2011_09_26-0005` должен будет лежать график зависимости размера карты от параметров одометрии

Сохранение всех этих данных уже реализовано в ячейках данного ноутбука. Достаточно просто выполнить пункт про визуализацию лидарной одометрии и запустить все ячейки.

Итого архив должен будет содержать:
```
hw_lidar_odometry.ipynb
hw_results/
    2011_09_26-0005/
        trajectories/
            trajectory_baseline.txt
            trajectory_baseline.png
            trajectory_LO_....txt
            trajectory_LO_....png
            ...
        visualizations/
            visualization_baseline.html    <- Визуализация краты для BASELINE-а
            visualization_LO_{?}_{?}.html  <- Визуализация карты для лучших параметров одометрии
            ...
        restored_map_sizes_for_50_cm.png  <- График зависимости размера карты от параметров одометрии
```

<a id='lo_versions'></a>
# Варианты лидарной одометрии<sup>[toc](#_toc)</sup>

<a id='lo_v1'></a>
## Лидарная одометрия (v1.0)<sup>[toc](#_toc)</sup>

Пусть к нам пришло очередное лидарное облако $\boldQ = [\boldq_1, \dots, \boldq_{M}]$ ($\boldq_i \in \RR^3$). Предыдущее лидарное облако обозначим как $\boldP = [\boldp_1, \dots, \boldp_N]$ ($\boldp_i \in \RR^3$). Требуется найти перемещение $\boldT$ лидара в пространстве относительно его предыдущей позы. Если говорить неформально, то это такой трансформ, который делает облако $\boldQ$ &laquo;похожим&raquo; на $\boldP$:
$$
\boldT \otimes \boldQ \approx \boldP.
$$
Эту задачу решает алгоритм ICP, описанный на лекции. Хорошее описание этого алгоритма также можно найти в документации `opend3d` https://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html.

В результате из последовательности лидарных облаков $\boldP_0$, $\boldP_1$, $\dots$, $\boldP_t$ получаем последовательность относительных перемещений лидара $\boldT_{1 \to 0}$, $\boldT_{2 \to 1}$, $\dots$, $\boldT_{i \to (i - 1)}$, $\dots$, $\boldT_{t \to (t - 1)}$. Здесь $\boldT_{i \to (i - 1)}$ обозначает трансформ, который представляет собой переход из системы координат лидарного облака $i$ в систему координат лидарного облака $i - 1$.

Наличие последовательность относительных перемещений лидара $\boldT_{1 \to 0}$, $\boldT_{2 \to 1}$, $\dots$, $\boldT_{i \to (i - 1)}$, $\dots$, $\boldT_{t \to (t - 1)}$ позволяет легко найти трансформ между любыми двумя облаками:
1. Трансформ из системы координат облака $\boldP_i$ в систему координат облака $\boldP_j$, где $i > j$, находится следующим образом:
    $$
    \boldT_{i \to j} = \boldT_{j \to (j - 1)} \otimes \dots \otimes \boldT_{(i - 1) \to (i - 2)} \otimes \boldT_{i \to (i - 1)}
    $$
2. Обратный трансформ находится как
    $$
    \boldT_{j \to i} = \boldT_{i \to j}^{-1} = \boldT_{i \to (i - 1)}^{-1} \otimes \dots \otimes \boldT_{j \to (j - 1)}^{-1}.
    $$


Обозначим через $\boldT_{0 \to \mathrm{world}}$ позу лидара в начальный момент времени относительно интересующей нас системы координат под названием $\mathrm{world}$. Это может быть, например, система координат в референсной точке проекции Меркатора. Ну или же вообще единичный трансформ, тогда мы по сути будем оценивать всю траекторию относительно начального положения лидара. В любом случае теперь мы можем выразить позу центрального лидара в момент времени $t$ относительно интересующей нас системы координат $\mathrm{world}$ через позу лидара в момент времени $t - 1$ отностиельно  $\mathrm{world}$:
$$
\boldT_{t \to \mathrm{world}} = \boldT_{t \to (t - 1)} \otimes \boldT_{(t - 1) \to \mathrm{world}}.
$$
Рекуррентно раскрывая $\boldT_{(t - 1) \to \mathrm{world}}$, получаем
$$
\boldT_{t \to \mathrm{world}} = \boldT_{t \to (t - 1)} \otimes \dots \otimes \boldT_{i \to (i - 1)} \otimes \dots  \otimes \boldT_{1 \to 0} \otimes \boldT_{0 \to \mathrm{world}}.
$$

Также мы можем взять все найденные положения лидара в пространстве $\{\boldT_{i \to \mathrm{world}}\}_{i=0}^t$ и смержить все облака в единое облако карты $\boldM_t$:
$$
\boldM_t = \sum\limits_{i=0}^t \boldT_{i \to \mathrm{world}} \boldP_t
$$

> <span style="color:blue">**Замечание.**</span> При слиянии облаков стоит разряжать карту, например, с помощью вокселизации, чтобы не было роста объема карты в ситуации, когда машина просто стоит.

<a id='lo_v2'></a>
## Лидарная одометрия (v2.0)<sup>[toc](#_toc)</sup>

У описанного выше алгоритма на практике проявляется основной недостаток &mdash; его низкая точность. Причин этому множество. Например, те же пустые пространства между колец в лидарном облаке, из-за которых при движении лидара значительная часть точек нового очередного облака может просто не попадать в окрестность точек предыдущего облака.


Поэтому рассмотрим небольшое расширение простейшей лидарной одометрии, в которой в каждый момент времени будем поддерживать буфер из $T$ предыдущих облаков, смерженных в карту. При поступлении очередного лидарного облака будем матчить его не к предыдущему, а именно к этой __on-the-fly-карте__. Затем добавляем новое облако в буфер, и попутно удаляем самое старое облако.

Пусть в момент времени $t$ к нам пришло облако $\boldP_t$, и мы каким-то обазом нашли его положение $\boldT_{t \to \mathrm{world}}$. Тогда в после матчинга обновляем on-the-fly-карту:
$$
\boldM_{[t - T + 1, t]} = \sum\limits_{i = t - T + 1}^{t} \boldT_{i \to \mathrm{world}} \otimes \boldP_t.
$$
Затем в момент времени $t + 1$ поступает облако $\boldP_{t + 1}$. Матчим его к облаку $\boldM_{[t - T + 1, t]}$, и находим трансформ $\boldT_{(t + 1) \to \mathrm{world}}$. Теперь добавляем облако $\boldP_{t+1}$ в карту и попутно удаляем облако $\boldP_{t - T + 1}$.


Фактически данный алгоритм действий эквивалентен простейшей лидарной одометрии в случае $T = 1$, т.е. когда в буфере всего одно облако.

<a id='lo_v3'></a>
## Лидарная одометрия (v3.0)<sup>[toc](#_toc)</sup>

Предыдущий алгоритм версии v2 страдает в ситации, когда машина долгое время стоит на месте, ведь в таком случае фактичеси мы придем к тому, что весь буфер будет представелен одним облаком. Затем, когда машина начнёт движение, мы вновь столкнёмся с проблемой пустых пространств между кольцами. С этим над что-то делать.

Довольно простое и логичное решение состоит в том, чтобы заменять облака в буфере не безусловно по мере их поступления, а когда, например, мы переместились относительно предыдущей позы (или любой из поз в буфере) на расстояние, превышающее пороговое значение $d$. 

**Собственно именно эту версию лидарной одометрии и предлагается реализовать**. 

Теперь немного о деталях реализиации:
1. Значение размера буфера $T$ и порога $d$ должно быть легко задать, чтобы перепрогнать одометрию. При $T = 1$ и $d = 0$ алгоритм должен превращаться простейшую лидарную одомерии версии v1.0.
2. При матчинге использует `point-to-plane ICP registration`. Тут всё придумано за нас, и самый простой способ &mdash; это взять примеры попарного матчинга облаков из [документации `open3d`](https://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html). Однако для этого нам нужны нормале в __target-облаке__, т.е. в облаке $\boldM_{[t - T + 1, t]}$.
3. Нормали также оцениваем с помощью `open3d`. Перед этим не забываем downsample-ить облака при слиянии &mdash; нам нет смысла в 100 точках в окрестности 5 см. Размер вокселя можно поставить оn 5 до 12.5 см. Радиус рассчета нормалей обычно выбирают в диапазоне 30 см до 50 см.
4. В итоге вокселизируем, считаем нормали, и получаем облако карты, подготовленное к регистрации нового облака лидара.
5. Находим положение нового облака относительно карты, обновляем буфер, пересчитываем карту, ждём следующего облака.

**Что должно быть на выходе?** 
1. Должна быт ьвозможность задать параметр $T$ и $d$ и перепрогнать алгоритм одометрии
2. Алгоритм одометрии должен сохранять найденный позы лидаров на диск. Например, можно сделать как-то так:
    ```
    hw/
      poses_T010_d100.txt  # < Размер буфера 10, d = 100см
      poses_T001_d000.txt  # < Размер буфера 1, d = 0см
    ```
3.  Должна быть ячейка кода, в которой можно указать значения параметров $T$ и $d$. Затем она подгрузит позы из файла (если он есть для таких значений) и смержит все облака согласно этим позам в единое облако. При слиянии карту следует вокселизировать с параметром `voxel_size` от 5см до 12.5см
4.  Затем должна быть ячейка визуализации. Тут также должен быть параметр вокселизации, но уже именно для того, чтобы визуализация не тормозила. Тут `voxel_size` может быть порядка 50см.

Проще говоря, должна быть возможность посмотреть на то, что получилось, и при необходимости перепрогнать

<a id='lo_v4'></a>
## Лидарная одометрия (v4.0)<sup>[toc](#_toc)</sup>

На самом деле можно реализовать лидарную одометрию с фактически неограниченным размером буфера $T = \infty$, т.е. когда мы вообще не удаляем отобранные облака из текущей карты, либо добавляем в неё все облака (проводя через downsampling). На практике основная проблема здесь &mdash; неограниченный рост карты. Если же говорить про наши реалиии в контексте датасета, то сложность матчинга и обновления такого облака будет расти со временем. Но все проблемы решаемы:
1. От неограниченного роста карты спасает умный dump кусков карты на диск, и загрузка с диска, когда потребуется (это если говорить про реалии робота).
2. Также можно реализовать хитрое обновление карты, сложность которого будет зависеть только от размера нового лидарного облака, но не от размера текущей карты.

Если скомбинировать оба подхода, то он окзаывается вполне приемлемым для того, чтобы робот во время проезда восстанаваливал карту местности, попутно локализуя себя в ней. Подобные алгоритмы называются _SLAM (Simultaneous Localization and Mapping)_.

In [ ]:
import typing as T
import os
import numbers
import logging
import copy
import collections
import numpy as np
from tqdm import trange
import matplotlib.pyplot as plt
%matplotlib inline

import pykitti
import open3d
import plotly.offline as py
py.init_notebook_mode(connected=True)

# PCL
from sdc.pcl.common.cloud_io import read_point_cloud_o3d, read_point_cloud_xyz
from sdc.pcl.common.transform_point_cloud import transform_point_cloud_xyz
from sdc.pcl.common.convert_point_cloud import (
    convert_point_cloud_o3d_to_xyz,
    convert_point_cloud_xyz_to_o3d,
)
from sdc.pcl.filters.voxel_grid import apply_voxel_grid
from sdc.pcl.filters.radius_outlier_removal import apply_radius_outlier_removal
from sdc.plotly.util import (
    create_plotly_figure,
    add_point_cloud,
    add_poses,
    add_positions,
    apply_min_max_scaling,
    convert_values_to_rgba_tuples_f64,
)

# TRANSFORMS
from sdc.transforms.transform_utils import (
    verify_transform_matrix,
    compose_transforms,
)
from sdc.transforms.convert_transform import (
    convert_rotation_matrix_to_quaternion,
    convert_quaternion_to_rotation_matrix,
)

# GEO
from sdc.geo.geo_lla_xyz_converter import GeoLlaXyzConverter
from sdc.geo.geo_position_lla import GeoPositionLLA
from sdc.geo.geo_position_xyz import GeoPositionXYZ

# KITTI
from sdc.kitti.dataset_adaptor import KittiDatasetAdaptor
from sdc.kitti.localization import (
    Localization,
    build_localization,
    build_localizations,
)
from sdc.kitti.convert_localization import (
    convert_localization_to_transform_matrix,
    convert_localizations_to_transform_matrices,
)

<a id='kitti'></a>
# KITTI<sup>[toc](#_toc)</sup>

## Загружаем датасет сцены<sup>[toc](#_toc)</sup>

### 1. Подключаемся к данным на диске (в датасете)<sup>[toc](#_toc)</sup>

In [ ]:
KITTI_DIR_PATH = os.path.abspath('./datasets/KITTI/')

# Указываем датасет для загрузки
RIDE_DATE = "2011_09_26"
DRIVE = "0106"  # Использовался в ДЗ 2025-го года
DRIVE = "0005"  # ДЗ 2026-го года. Нужно использовать его в финальной отправке

# Загружаем данные. Опционально можно указать диапазон фреймов для загрузки
KITTI_DATASET = pykitti.raw(base_path=KITTI_DIR_PATH, date=RIDE_DATE, drive=DRIVE)

# Сразу же создаем адаптор
KITTI_DATASET_ADAPTOR = KittiDatasetAdaptor(KITTI_DATASET)
NUM_FRAMES = KITTI_DATASET_ADAPTOR.num_lidar_clouds

print('Dataset info:')
print(f'\tdirectory: <{KITTI_DIR_PATH}>')
print(f'\tnum_velodyne_clouds: {NUM_FRAMES}')

### 2. Локализация и позы<sup>[toc](#_toc)</sup>

#### Формируем показания локализации<sup>[toc](#_toc)</sup>

Датасет KITTY содержит оценки глобальной + ориентации. На основании эти данных ниже мы формируем показания локализации (см. класс `Localization`). Далее эти показания локализации будут преобразованы в матрицы размера 4x4, описывающие переход из системы координат машины в некоторую референсную систему координат, которую на лекции/семинаре мы называли `world`.

In [ ]:
KITTI_DATASET_ADAPTOR = KittiDatasetAdaptor(KITTI_DATASET)
reference_point = KITTI_DATASET_ADAPTOR.read_geo_positions_lla()[0]
localizations: T.List[Localization] = KITTI_DATASET_ADAPTOR.build_localizations(reference_point)

del reference_point
assert len(localizations) == NUM_FRAMES
print(f'num_localizations: {len(localizations)}')

#### Формируем позы в системе координат меркатора<sup>[toc](#_toc)</sup>

In [ ]:
gps_to_mercator_transforms = convert_localizations_to_transform_matrices(localizations)
print(f'Number of poses: {len(gps_to_mercator_transforms)}')

In [ ]:
# Визуализируем все позиции машины
fig = create_plotly_figure()
add_positions(fig, [T[:3, 3] for T in gps_to_mercator_transforms])
fig.show()

In [ ]:
# Визуализириуем первые 60 поз машины, т.е. будет визуализирована ориентация в виде отрисовки осей
# При попытке визуализации бОльшего числа поз plotly может подвиснуть
fig = create_plotly_figure()
add_poses(fig, gps_to_mercator_transforms[:60])
fig.show()

### 3. Облака велодайна<sup>[toc](#_toc)</sup>

Чем лучше начальное приближение, тем лучше будет сходимость ICP. У нас есть два простых способа задать начальный трансформ между двумя последовательными лидарными облаками:
1. Единичный трансформ. Скорее всего будет не очень хорошим приближением в случае быстрого движения машины.
2. На основе показаний локализации, где в качестве начального трансформа можно взять преобразование (трансформ) из source-позы в target-позу.

Но довольно слов, давайте просто посмотрим на оба варианта. В ячейке кода ниже мы берем три последовательных облака и отрисовываем их либо как есть, т.е. предполагая единичный трансформ, либо же приводим их в единую систему координат world (относительно которой заданы позы в `poses`). Первый или второй варинат задаются параметром `apply_to_world_transform`.

> **Внимание**. В случае проезда `DRIVE = "0106"` движение машины начинается только где-то с 50-го фрейма. До этого момента она или практически неподвижна. Ниже специально был подобран фрейм 90, чтобы продемонстрировать разницу между начальными приближениями. Если работа происходит с другим проездом, то параметр `init_frame_idx` возможно потребуется изменить.

Запустите ячейку ниже с обоими значениями `apply_to_world_transform`, чтобы увидеть разницу:
1. В случае `apply_to_world_transform=False` вы должны увидеть явное двоение/невыровненность облаков друг относительно друга
2. В случае `apply_to_world_transform=True` облака приводятся в единую систему координат с помощью показаний локализации, и в таком случае двоение должно быть сильно меньше

In [ ]:
init_frame_idx = 90

# Используем вокселизацию для уменьшения размера облака, чтобы отрисовка не тормозила
voxel_size = 0.15

# Привести к единой системе координат world (на основе показаний GPS-а), или нет?
apply_transform_to_world = True

clouds_xyz = []
for frame_idx in range(init_frame_idx, init_frame_idx + 3):
    cloud_xyz = KITTI_DATASET_ADAPTOR.read_lidar_cloud_xyzi(frame_idx)[..., :3]
    if apply_transform_to_world:
        # Приводим облако к референсной системе координат world
        cloud_xyz = transform_point_cloud_xyz(
            cloud_xyz.astype(np.float64),
            gps_to_mercator_transforms[frame_idx]).astype(np.float32)
    initial_cloud_size = cloud_xyz.shape[0]
    cloud_xyz = apply_voxel_grid(cloud_xyz, voxel_size)  # Сэмплировование облака, чтобы отрисовка не зависала
    final_cloud_size = cloud_xyz.shape[0]
    print(f'{frame_idx}: {initial_cloud_size} -> voxel_grid(voxel_size={voxel_size}) -> {final_cloud_size}')
    clouds_xyz.append(cloud_xyz)
    del cloud_xyz

fig = create_plotly_figure(bgcolor='black')
add_point_cloud(fig, clouds_xyz[0], colors='red')
add_point_cloud(fig, clouds_xyz[1], colors='green')
add_point_cloud(fig, clouds_xyz[2], colors='blue')
fig.show()

del clouds_xyz

<a id='lo_impl'></a>
# Реализация лидарной одометрии<sup>[toc](#_toc)</sup>
* [Подготовительные действия](#lo_impl_prep)
* [Основная работа](#lo_impl_main)
* [Визуализация результатов](#lo_vis)

In [ ]:
from enum import StrEnum

class AlgorithmType(StrEnum):
    BASELINE = "baseline"
    LIDAR_ODOMETRY = "LO"  # lidar odometry

<a id='lo_impl_prep'></a>
## Подготовительные действия<sup>[toc](#_toc)</sup>

У нашей лидарной одометрии два основных параметра:
* `max_lidar_clouds_buffer_size` &mdash; максимальное количество облаков в текущем буфере
* `min_distance_between_clouds_cm` &mdash; минимальное расстояние нового облака относительно предыдущего, при котором добавляем его в буфер (попутно удаляя самое старое облако из того же буфера). Расстояние указываем в сантиметрах

In [ ]:
# Два основных параметра, описывающих нашу лидарную одометрию
max_lidar_clouds_buffer_size: int = 10
min_distance_between_clouds_cm: int = 100  # In [cm]eters
del max_lidar_clouds_buffer_size, min_distance_between_clouds_cm  # Пока удаляем

Всего облаков у нас 227. Поэтому нет смысла указывать размер буфера больше, чем количество облаков:

In [ ]:
MAX_LIDAR_CLOUDS_BUFFER_SIZE = KITTI_DATASET_ADAPTOR.num_lidar_clouds

### Сохранение и загрузка результатов<sup>[toc](#_toc)</sup>

Результаты для конкретного проезда будем хранить в отдельно директории `hw_results/{RIDE_DATE}-{DRIVE}` в следующем формате:
```
{current workding dir}/
hw_results/
    2011_09_26-0005/
        trajectories/
            trajectory_baseline.txt
            trajectory_lo_1_100.txt
            trajectory_lo_10_200.txt
            ...
        visualizations/
            visualization_baseline.html
            visualization_lo_1_100.html
            visualization_lo_10_200.html
            ...
```

In [ ]:
DRIVE_RESULTS_DIR_PATH = os.path.abspath(f'hw_results/{RIDE_DATE}-{DRIVE}')
TRAJECTORIES_DIR_PATH = os.path.join(DRIVE_RESULTS_DIR_PATH, 'trajectories')
os.makedirs(TRAJECTORIES_DIR_PATH, exist_ok=True)
TRAJECTORY_FILE_NAME_PREFIX = 'trajectory'
TRAJECTORY_FILE_NAME_EXT = 'txt'

VISUALIZATIONS_DIR_PATH = os.path.join(DRIVE_RESULTS_DIR_PATH, 'visualizations')
os.makedirs(VISUALIZATIONS_DIR_PATH, exist_ok=True)
VISUALIZATION_FILE_NAME_PREFIX = 'visualization'
VISUALIZATION_FILE_NAME_EXT = 'html'

print(f'Results for drive {RIDE_DATE}/{DRIVE} will be stored at:\n\t<{DRIVE_RESULTS_DIR_PATH}>')
print(f'Trajectories will be stored at:\n\t<{TRAJECTORIES_DIR_PATH}>')
print(f'Visualizations will be stored at:\n\t<{VISUALIZATIONS_DIR_PATH}>')

**В ячейке ниже расположены вспомогательные функции для работы именами файлов в директории с результатам**

In [ ]:
def build_results_file_name(
        file_name_prefix: str,
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int,
        file_ext: str) -> str:
    assert os.path.basename(file_name_prefix) == file_name_prefix
    assert os.path.splitext(file_name_prefix)[1] == ''
    if algorithm_type == AlgorithmType.BASELINE:
        return '{}_{}.{}'.format(file_name_prefix, algorithm_type, file_ext)

    assert isinstance(max_lidar_clouds_buffer_size, numbers.Integral)
    assert isinstance(min_distance_between_clouds_cm, numbers.Integral)
    assert 0 <= max_lidar_clouds_buffer_size <= MAX_LIDAR_CLOUDS_BUFFER_SIZE
    assert 0 <= min_distance_between_clouds_cm
    assert file_ext in ['txt', 'html']
    return '{}_{}_{}_{}.{}'.format(
        file_name_prefix,
        algorithm_type,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm,
        file_ext)


def get_trajectory_txt_file_name(
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return build_results_file_name(
        TRAJECTORY_FILE_NAME_PREFIX,
        algorithm_type,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm,
        TRAJECTORY_FILE_NAME_EXT
    )

def get_trajectory_png_file_name(
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return build_results_file_name(
        TRAJECTORY_FILE_NAME_PREFIX,
        algorithm_type,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm,
        "png",
    )


def get_trajectory_png_file_path(
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return os.path.join(
        TRAJECTORIES_DIR_PATH,
        get_trajectory_png_file_name(algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))


def get_visualization_html_file_name(
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return build_results_file_name(
        VISUALIZATION_FILE_NAME_PREFIX,
        algorithm_type,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm,
        VISUALIZATION_FILE_NAME_EXT)


def get_trajectory_txt_file_path(
        algorithm_type: AlgorithmType,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return os.path.join(
        TRAJECTORIES_DIR_PATH,
        get_trajectory_txt_file_name(algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))


def get_visualization_html_file_path(
        algorithm_type,
        max_lidar_clouds_buffer_size: int,
        min_distance_between_clouds_cm: int) -> str:
    return os.path.join(
        VISUALIZATIONS_DIR_PATH,
        get_visualization_html_file_name(algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))


def parse_results_file_name(file_name: str) -> T.Tuple[AlgorithmType, T.Optional[T.Tuple[int, int]]]:
    assert file_name.startswith(TRAJECTORY_FILE_NAME_PREFIX) or\
        file_name.startswith(VISUALIZATION_FILE_NAME_PREFIX), file_name
    basename, ext = os.path.splitext(file_name)
    assert ext == '.' + TRAJECTORY_FILE_NAME_EXT or\
        ext == '.' + VISUALIZATION_FILE_NAME_EXT

    if AlgorithmType.BASELINE in basename:
        assert len(basename.split('_')) == 2
        return AlgorithmType.BASELINE, None

    assert AlgorithmType.LIDAR_ODOMETRY in basename
    max_lidar_clouds_buffer_size, min_distance_between_clouds_cm = basename.split('_')[-2:]
    max_lidar_clouds_buffer_size = int(max_lidar_clouds_buffer_size)
    min_distance_between_clouds_cm = int(min_distance_between_clouds_cm)
    return AlgorithmType.LIDAR_ODOMETRY, (max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)

Пример значений имен и путей к файлам результатов:

In [ ]:
algorithm_type = AlgorithmType.LIDAR_ODOMETRY
max_lidar_clouds_buffer_size = 123
min_distance_between_clouds_cm = 456

trajectory_txt_file_name = get_trajectory_txt_file_name(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)
trajectory_txt_file_path = get_trajectory_txt_file_path(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)

visualization_html_file_name = get_visualization_html_file_name(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)
visualization_html_file_path = get_visualization_html_file_path(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)

print('Example of results files names for algorithm = {}, T = {}, d = {} [cm]:'.format(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))
print(f'\tresults TXT file name:        <{trajectory_txt_file_name}>')
print(f'\tvisualization HTML file name: <{visualization_html_file_name}>')

print('\nExample of results files paths for algorithm = {}, T = {}, d = {} [cm]:'.format(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))
print(f'\tresults TXT file path:        <{trajectory_txt_file_path}>')
print(f'\tvisualization HTML file path: <{visualization_html_file_path}>')

del max_lidar_clouds_buffer_size, min_distance_between_clouds_cm
del trajectory_txt_file_name, visualization_html_file_name
del trajectory_txt_file_path, visualization_html_file_path

Посмотрим на то, какие результаты уже есть в папке. Изначально никаких результатов нет, что и ожидаем увидеть при первом запуске.

In [ ]:
os.makedirs(TRAJECTORIES_DIR_PATH, exist_ok=True)
print(f'Lidar odometry results are stored in <{TRAJECTORIES_DIR_PATH}>')
results_file_names = sorted(os.listdir(TRAJECTORIES_DIR_PATH))
if len(results_file_names) == 0:
    print('No currently present results')
else:
    print('Currently presents results:')
    for results_file_name in sorted(os.listdir(TRAJECTORIES_DIR_PATH)):
        algorithm_type, algorithm_params  = parse_results_file_name(results_file_name)
        if algorithm_type == AlgorithmType.LIDAR_ODOMETRY:
            max_lidar_clouds_buffer_size, min_distance_between_clouds_cm = algorithm_params
            print('\t{} -> algorithm = {}, T = {}, d = {} [cm]'.format(
                results_file_name, algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm))
            del max_lidar_clouds_buffer_size, min_distance_between_clouds_cm
        else:
            print('\t{} -> algorithm = {}'.format(results_file_name, algorithm_type))
    del algorithm_type, results_file_name
del results_file_names

**Для сохранения найденных поз/трансформов в файл lidar_trajectory...txt и их считывания оттуда будем использовать функции ниже:**

In [ ]:
def write_transforms_matrices_to_file(file_path: str, transforms_matrices: T.List[np.ndarray]):
    with open(file_path, 'w') as f:
        f.write('# QX QY QZ QW TX TY TZ TW\n')
        for transform_matrix in transforms_matrices:
            verify_transform_matrix(transform_matrix)
            qx, qy, qz, qw = convert_rotation_matrix_to_quaternion(transform_matrix[:3, :3])
            tx, ty, tz = transform_matrix[:3, 3]
            f.write('{} {} {} {} {} {} {}\n'.format(qx, qy, qz, qw, tx, ty, tz))


def read_transforms_matrices_from_file(file_path: str) -> T.List[np.ndarray]:
    transforms_matrices: T.List[np.ndarray] = []
    with open(file_path, 'r') as f:
        for line in f.readlines():
            if line.startswith('#'):
                continue
            qx, qy, qz, qw, tx, ty, tz = [float(v) for v in line.split()]
            rotation_matrix = convert_quaternion_to_rotation_matrix([qx, qy, qz, qw])
            translation_vector = np.array([tx, ty, tz])
            transform_matrix = np.eye(4, dtype=np.float64)
            transform_matrix[:3, :3] = rotation_matrix
            transform_matrix[:3, 3] = translation_vector
            transforms_matrices.append(transform_matrix)
    return transforms_matrices

<a id='lo_impl_main'></a>
## Основная работа<sup>[toc](#_toc)</sup>
* [Normals estimation](#normals_estimation)
* [Point cloud registration](#point_cloud_registration)
* [Запуски лидарной одометрии](#lidar_odometry_runs)

<a id='normals_estimation'></a>
### Normals estimation<sup>[toc](#_toc)</sup>

При сопосоставлении облаков мы будем минимизировать ICP-метрику point-to-plane. А это значит, что нам потребуется уметь оценивать нормали target-облака. И если в этой процедуре совершить ошибку, то ничего хорошего матчинг не покажет.

Нормаль для некоторой точки $\boldp_i$ облака $\boldP = [\boldp_1, \dots, \boldp_N]$ оценивается по некоторой окрестности самой точки $\boldp_i$, т.е. находятся ближайшие соседей $\boldp_i$, и через них проводится плоскость. Нормаль данной плоскости и берется в качестве нормали точки $\boldp_i$. По сути параметры поиска нормали представлюят собой параметры поиска ближайших соседей:

Ссылки на документацию open3d:
* https://www.open3d.org/docs/0.7.0/python_api/open3d.geometry.estimate_normals.html
* https://www.open3d.org/docs/0.7.0/python_api/open3d.geometry.KDTreeSearchParam.html

Есть три типа параметров для поиска соседей с помощью KD-дерева:

In [ ]:
print(open3d.geometry.KDTreeSearchParam.Type.KNNSearch)
print(open3d.geometry.KDTreeSearchParam.Type.RadiusSearch)
print(open3d.geometry.KDTreeSearchParam.Type.HybridSearch)

Сами классы параметров:

In [ ]:
open3d.geometry.KDTreeSearchParam   # Базовый класс
open3d.geometry.KDTreeSearchParamKNN
open3d.geometry.KDTreeSearchParamRadius
open3d.geometry.KDTreeSearchParamHybrid

1. Всегда берем $K$ ближайших соседей:

In [ ]:
search_param = open3d.geometry.KDTreeSearchParamKNN(knn=30)
print(search_param.get_search_type())
print(search_param.knn)

2. Берем всех соседей в радиусе $R$:

In [ ]:
search_param = open3d.geometry.KDTreeSearchParamRadius(radius=0.5)
print(search_param.get_search_type())
print(search_param.radius)

3. Гибридная логика поиска:

In [ ]:
search_param = open3d.geometry.KDTreeSearchParamHybrid(radius=0.5, max_nn=30)
print(search_param.get_search_type())
print(search_param.radius)
print(search_param.max_nn)

**Функция для рассчета нормалей по облаку `open3d.geometry.PointCloud`:**

In [ ]:
def estimate_normals(
        point_cloud_o3d: open3d.geometry.PointCloud,
        search_param: open3d.geometry.KDTreeSearchParam) -> open3d.geometry.PointCloud:
    poitn_cloud_o3d = copy.deepcopy(point_cloud_o3d)
    point_cloud_o3d.estimate_normals(search_param)
    return point_cloud_o3d

<a id='point_cloud_registration'></a>
### Point cloud registration<sup>[toc](#_toc)</sup>

Ссылки на документацию:
* https://www.open3d.org/docs/release/tutorial/pipelines/icp_registration.html

**Функция для нахождения трансформа между source-облаком XYZ и target-облаком с нормалями:**

In [ ]:
def align_point_clouds(
        source_point_cloud: open3d.geometry.PointCloud,
        target_point_cloud: open3d.geometry.PointCloud,
        init_source_to_target_transformation: np.ndarray,
        max_correspondence_distance: float) -> np.ndarray:
    assert max_correspondence_distance > 0.2, "Usage of very small distance (20 cm). You can simply remove this check "\
        "if you are sure what you are doing"
    registration = open3d.pipelines.registration.registration_icp(
        source=source_point_cloud,
        target=target_point_cloud,
        max_correspondence_distance=max_correspondence_distance,
        init=init_source_to_target_transformation,
        estimation_method=open3d.pipelines.registration.TransformationEstimationPointToPlane())
    return registration.transformation

<span style="color:red">Внимание, есть ряд важных замечаний:</span>
1. Если target-облако не будет обладать нормалями, то функция упадет с exception-ом. Всё логично, раз уж в `estimation_method`-е просим использовать метрику point-to-plane, для которой нужны нормали в target-облаке (в source-облаке не нужны), то извольте их предоставить.
2. Изучите внимательно смысл параметра `max_correspondence_distance` &mdash; это по сути радиус поиска соседей при подсчете correspondence-ов (т.е. соответствий), о которых рассказывалось на лекции. Очень частой ошибкой было то, что при выполнении домашней работы брали значения из документации, которые были равны нескольким сантиметрам, но в документации open3d используется пример совсем иного масштаба. В нашем случае требуется значения порядка 0.5 метра и выше. На практике при построении карт встречаются и значения порядка 5 метров, когда нужно строить длинные связи между облаками. Такая ситуация встречается, когда в некоторой зоне построения карт была низкая точность локализации, из-за чего `init_source_to_target_transformation`, полученный на основе EKF-локализации оказывался крайне неточен.

<a id='lidar_odometry_runs'></a>
### Запуски лидарной одометрии<sup>[toc](#_toc)</sup>

Результатом выполнения кода в ячейках ниже должен быть список `lidar_trajectory` из поз центрального лидара в системе координат, связанной с первым положением лидара. 

Требуется запустить лидарную одометрию для следующих параметров:
1. **Лидарная одометрия версии 1.0**:
    * `max_lidar_clouds_buffer_size = 1`
    * `min_distance_between_clouds_cm = 0`
    * Всегда всегда заменяем предыдущее облако новым
2. **Лидарная одометрия версии 2.0**:
    * `max_lidar_clouds_buffer_size = 10`
    * `min_distance_between_clouds_cm = 0`
    * Держим буффер из максимум 10 предыдущих облаков и всегда вытесняем самое старое облако новым
3. **Лидарная одометрия версии 3.0**:
    * `max_lidar_clouds_buffer_size = 10`
    * `min_distance_between_clouds_cm = 100`
    * Держим буффет из максимум 10 предыдущих облаков и вытесняем самое старое облако, только если сдвинулись отностиельно текущей позиции на $\geqslant 100$ сантиметров
5. То же самое, что и в предыдущем пункте, но теперь для диапазона значений `max_lidar_clouds_buffer_size` от 1 до 20 (для 10 уже подсчитали в предыдущем пункте). Это нам потребуется чуть дальше, чтобы оценить, насколько лучше становится наша восстановленная карта в зависимости от параметров одометрии. Что касается параметра `min_distance_between_clouds_cm`, то можно либо просто оставить 1 метр, либо поставить что-то свое большее нуля, но главное, чтобы он был одинаковый для всех размеров буфера.

In [ ]:
def run_lidar_odometry(
    algorithm_type: AlgorithmType,
    initial_lidar_trajectory: T.List[np.ndarray],
    max_lidar_clouds_buffer_size: int,
    min_distance_between_clouds_cm: int,
    **kwargs,
) -> T.List[np.ndarray]:
    """
    :param initial_lidar_trajectory: Предполагаемая траектория 
    """
    assert isinstance(algorithm_type, AlgorithmType)
    lidar_trajectory = []
    
    if algorithm_type == AlgorithmType.BASELINE:
        # Baseline
        # В качестве baseline-решения возвращаем тректорию на основании локализации:
        # ищем положения всех облаков относительно положения первого облака
        init_cloud_to_world_transform = gps_to_mercator_transforms[0]
        world_to_init_cloud_transform = np.linalg.inv(init_cloud_to_world_transform)
        for cloud_to_world_transform in gps_to_mercator_transforms:
            lidar_trajectory.append(world_to_init_cloud_transform @ cloud_to_world_transform)
        return lidar_trajectory

    # TODO Implement lidar odometry
    raise NotImplementedError(f'Algorithm "{algorithm_type}" not implemented')
    return lidar_trajectory

#### Пункты 1-3. Единичные запуски лидарной одометрии<sup>[toc](#_toc)</sup>

<span style="color:red">**Внимание**</span>, ниже для запуска ячейки по умолчанию используется baseline-решение (`AlgorithmType.BASELINE`), так как на момент первого запуска у вас ещё не написана реализация одометрии. Впоследствии вам надо будет поменять на алгоритм `AlgorithmType.LIDAR_ODOMETRY`.

In [ ]:
# 1. Параметры для запуска одометрии
algorithm_type = AlgorithmType.BASELINE
# algorithm_type = AlgorithmType.LIDAR_ODOMETRY
max_lidar_clouds_buffer_size: int = 10
min_distance_between_clouds_cm: int = 50  # Внимание, в матрицах трансформов лежат метры. Не забудьте отмасштабировать на 100

# 2. Файл для сохранения финальной траектории
trajectory_txt_file_path = get_trajectory_txt_file_path(
    algorithm_type, 
    max_lidar_clouds_buffer_size,
    min_distance_between_clouds_cm)
print(f'Results will be saved to file: <{trajectory_txt_file_path}>')
# Проверка, чтобы случайно не перетереть результаты.
# Если результаты нужно переписать, то следует закомментировать assert
rewrite_results = False
assert not os.path.isfile(trajectory_txt_file_path) or rewrite_results,\
    f'File <{os.path.basename(trajectory_txt_file_path)}> already exists. '\
    'Please disable this check manually or force rewrite file if you want to recalculate odometry'

# 3. Параметры для оценки нормалей и запуска ICP. Далеко не факт, что они являются наилучшими.
# Здесь просто для примера поставлены такими. Более того, можете спокойно добавлять какие-либо другие параметры, если потребуется
map_voxel_size = 0.05
map_normals_radius_search = 0.5
max_correspondence_distance = 0.5

# 4. Запуск лидарной одометрии
# TODO: Реализовать лидарную одометрию. Результатом выполнения данной ячейки должен
# быть список из матриц размера 4x4, описывающих положение облаков относительно самого первого облака
lidar_trajectory: T.List[np.array] = run_lidar_odometry(
    algorithm_type=algorithm_type,
    initial_lidar_trajectory=gps_to_mercator_transforms,
    max_lidar_clouds_buffer_size=max_lidar_clouds_buffer_size,
    min_distance_between_clouds_cm=min_distance_between_clouds_cm,
    map_voxel_size=map_voxel_size,
    map_normals_radius_search=map_normals_radius_search,
    max_correspondence_distance=max_correspondence_distance)
assert len(lidar_trajectory) == len(gps_to_mercator_transforms)

# 5. Сохраняем полученную траекторию
print(f'Saving trajectory to <{trajectory_txt_file_path}>')
write_transforms_matrices_to_file(trajectory_txt_file_path, lidar_trajectory)
del trajectory_txt_file_path

> Так как реализация функции `run_lidar_odometry`, которая и занимется нахождением траектории `lidar_trajectory`, является домашней работой, а нам нужно пока что как-нибудь позапускать следующие ячейки, то поступим следующим образом: инициализируем этот список единичными трансформами или же трансформами на основе траектории GPS-сенсора в системе меркатора (из проезда). Это позволит нам далее запустить блоки с сохранением, загрузкой и визуализацией результатов.

#### Пункт 4. Запуск лидарной одометрии с различными значениями `max_lidar_clouds_buffer_size`<sup>[toc](#_toc)<sup>

Если реализована функция `run_lidar_odometry` для алгоритма `AlgorithmType.LIDAR_ODOMETRY`, то тут остается лишь простой цикл, который бежит по различным значениями `max_lidar_clouds_buffer_size` и запускает дли них лидарную одометрию:

In [ ]:
algorithm_type = AlgorithmType.LIDAR_ODOMETRY
min_distance_between_clouds_cm = 50
map_voxel_size = 0.05
map_normals_radius_search = 0.5
max_correspondence_distance = 0.5

for max_lidar_clouds_buffer_size in range(1, 21):
    trajectory_txt_file_path = get_trajectory_txt_file_path(
        algorithm_type,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm)
    if os.path.isfile(trajectory_txt_file_path):
        print(f'Already exists <{trajectory_txt_file_path}>')
        continue
    lidar_trajectory = run_lidar_odometry(
        algorithm_type,
        gps_to_mercator_transforms,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm,
        map_voxel_size=map_voxel_size,
        map_normals_radius_search=map_normals_radius_search,
        max_correspondence_distance=max_correspondence_distance)
    assert len(lidar_trajectory) > 0, 'Please check that lidar odometry is implemented'
    print(f"Writing results to file <{trajectory_txt_file_path}>")
    write_transforms_matrices_to_file(trajectory_txt_file_path, lidar_trajectory)
    del trajectory_txt_file_path, lidar_trajectory

del min_distance_between_clouds_cm
del map_voxel_size
del map_normals_radius_search
del max_correspondence_distance

<a id='lo_vis'></a>
## Анализ результатов<sup>[toc](#_toc)</sup>

Здесь нужно выполнить два пункта:
1. [Визуализировать лидарную карту местности для конкретных параметров одометрии](#lo_vis_restored_map)
2. [Построить график размера лидарной карты от параметров одометрии](#lo_vis_restored_map_sizes)

<a id='lo_vis_restored_map'></a>
### 1. Визуализация восстановленной карты для конкретных параметров алгоритма<sup>[toc](#_toc)</sup>

В данном блоке просто проверяем себя, что то, что получили, смотрится адекватно

#### Выбираем параметры для визуализации<sup>[toc](#_toc)</sup>

In [ ]:
algorithm_type = AlgorithmType.BASELINE
# algorithm_type = AlgorithmType.LIDAR_ODOMETRY
max_lidar_clouds_buffer_size: int = 10
min_distance_between_clouds_cm: int = 50

trajectory_txt_file_path = get_trajectory_txt_file_path(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)
print(f'<{trajectory_txt_file_path}>')
assert os.path.isfile(trajectory_txt_file_path), "File with results does not exist. Did you run lidar odometry for selected parameters?"

#### Загружаем траекторию из файла<sup>[toc](#_toc)</sup>

In [ ]:
print(f'Loading trajectory from <{trajectory_txt_file_path}>')
lidar_trajectory = read_transforms_matrices_from_file(trajectory_txt_file_path)
del trajectory_txt_file_path
print(f'Loaded {len(lidar_trajectory)} poses/transforms')

#### Визуализируем траекторию<sup>[toc](#_toc)</sup>

Просто визуализируем, как выглядит найденная траектория машины/лидара при взгляде сверху.

In [ ]:
extend_coff = 0.1
lidar_positions = np.array([lidar_pose[:3, 3] for lidar_pose in lidar_trajectory])
lidar_positions_x = lidar_positions[:, 0]
lidar_positions_y = lidar_positions[:, 1]
print(f'x_range: {np.min(lidar_positions_x), np.max(lidar_positions_x)}')
print(f'y_range: {np.min(lidar_positions_y), np.max(lidar_positions_y)}')

x_min_index = np.argmin(lidar_positions_x)
x_max_index = np.argmax(lidar_positions_x)
x_min = lidar_positions_x[x_min_index]
x_max = lidar_positions_x[x_max_index]
x_span = x_max - x_min
print(f'x_min=x[{x_min_index}]={x_min}')
print(f'x_max=x[{x_max_index}]={x_max}')

y_min_index = np.argmin(lidar_positions_y)
y_max_index = np.argmax(lidar_positions_y)
y_min = lidar_positions_y[y_min_index]
y_max = lidar_positions_y[y_max_index]
y_span = y_max - y_min
print(f'y_min=y[{y_min_index}]={y_min}')
print(f'y_max=y[{y_max_index}]={y_max}')

plt.figure(figsize=(18, 6))
plt.axis('equal')
plt.scatter(lidar_positions_x, lidar_positions_y, color='b', alpha=0.5, edgecolor='k')
plt.xlim(x_min - extend_coff * x_span, x_max + extend_coff * x_span)
plt.ylim(y_min - extend_coff * y_span, y_max + extend_coff * y_span)
plt.xlabel('x', fontsize=14)
plt.ylabel('y', fontsize=14)
plt.title('lidar trajectory', fontsize=16)
plt.grid(which='both', linestyle='--', alpha=0.5)

trajectory_png_path = get_trajectory_png_file_path(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)
print(f'Saving trajectory image to <{trajectory_png_path}>')
plt.savefig(trajectory_png_path, dpi=300)

#### Восстанавливаем лидарную карту<sup>[toc](#_toc)</sup>

Просто сливаем воедино все облака согласно предоставленной траектории. Также вокселизируем полученное мега-облако с параметром `map_voxel_size`.

In [ ]:
def restore_map(lidar_poses: T.List[np.ndarray], map_voxel_size: float, frames_range=None):
    assert len(lidar_poses) == NUM_FRAMES

    if frames_range is None:
        frames_range = range(NUM_FRAMES)

    # Переводим облака в единую систему координат
    transformed_clouds_xyz = []
    for frame_idx in frames_range:
        cloud_xyz = KITTI_DATASET_ADAPTOR.read_lidar_cloud_xyz(frame_idx)
        transformed_cloud_xyz = transform_point_cloud_xyz(
            cloud_xyz.astype(np.float64),
            lidar_poses[frame_idx]).astype(np.float32)
        transformed_clouds_xyz.append(transformed_cloud_xyz)

    merged_cloud_xyz = np.vstack(transformed_clouds_xyz)
    del transformed_clouds_xyz

    if map_voxel_size > 0.:
        merged_cloud_xyz = apply_voxel_grid(merged_cloud_xyz, map_voxel_size)

    return merged_cloud_xyz


def get_restored_map_size(lidar_poses: T.List[np.ndarray], map_voxel_size: float, frames_range=None):
    return restore_map(lidar_poses, map_voxel_size, frames_range).shape[0]

##### Создаем единое смерженное облако + вокселизируем<sup>[toc](#_toc)</sup>

In [ ]:
voxel_sizes = [0.5, 0.125]

# Чтобы не перегружать визуализацию, можно ограничить список облаков, которые будут смержены
frames_range = range(90, 120)
# frames_range = None  # Смержить все облака

merged_cloud_xyz = restore_map(lidar_trajectory, map_voxel_size=-1., frames_range=frames_range)
print(f'Number of points in merged cloud (no voxelization): {merged_cloud_xyz.shape[0]}', flush=True)

for voxel_size in voxel_sizes:
    voxelized_merged_cloud_xyz = apply_voxel_grid(merged_cloud_xyz, voxel_size)
    print(f'Number of points after voxelization with voxel_size={voxel_size} [m]: {voxelized_merged_cloud_xyz.shape[0]}', flush=True)
    del voxel_size, voxelized_merged_cloud_xyz
del merged_cloud_xyz

#### Визуализируем смерженное облако<sup>[toc](#_toc)</sup>

In [ ]:
map_voxel_size = 0.5
min_frame_idx = 90
max_frame_idx = 120
frames_range = range(min_frame_idx, max_frame_idx)
frames_range = None  # Смержить все облака

merged_cloud_xyz = restore_map(lidar_trajectory, map_voxel_size, frames_range)
del map_voxel_size, min_frame_idx, max_frame_idx

fig = create_plotly_figure(bgcolor='black')
fig.update_layout(
    title_text=f'Merged cloud size = {merged_cloud_xyz.shape[0]}',
    title_font_size=16,
    title_automargin=True)

colors = apply_min_max_scaling(np.clip(merged_cloud_xyz[:, 2], -2, 7))
colors = colors**(2/3.)  # Гамма-коррекция цветов 
colors = convert_values_to_rgba_tuples_f64(colors, cmap='gist_rainbow')

add_point_cloud(fig, merged_cloud_xyz, colors=colors)
fig.show()

visualization_html_file_path = get_visualization_html_file_path(
    algorithm_type, max_lidar_clouds_buffer_size, min_distance_between_clouds_cm)
print(f'Saving HTML to <{visualization_html_file_path}>')
fig.write_html(visualization_html_file_path)
del visualization_html_file_path

#### Что должно получиться?

Ниже визуализирована карта для проезда `DRIVE="0106"` для `BASELINE`-а и для `LIDAR_ODOMETRY` с некоторыми параметрами. Даже при первом взгляде становится понятно, что `BASELINE`-алгоритм приводил к сильным двоениями и "пушистости" карты. Похожая картина должна получиться и у вас на датасете `DRIVE="0005"`. В принципе никто не мешает при отладке алгоритма выбрать любой из проездов. Главное, чтобы финальная версия в ноутбуке была для `"0005"`.

У вас в результатах должно быть как минимум две визуализации:
1. Одна для `BASELINE`-а
2. Вторая для наилучших параметров `LIDAR_ODOMETRY` (про их поиск см. следующий пункт)

In [ ]:
from IPython.display import Image

In [ ]:
Image('./images/hw_results_baseline.png', width='100%')

In [ ]:
Image('./images/hw_results_lo.png', width='100%')

<a id='lo_vis_restored_map_sizes'></a>
### 2. График размера карты от параметров одометрии<sup>[toc](#_toc)</sup>

В предыдущем пункте при визуализации мы мержили лидарные облака согласно их позам, и вокселизировали смерженное облако, тем самым получая некоторую карту местности. Причем интуитивно можем определить качество карты, как размер полученного облака, так как чем лучше карта, тем она менее &laquo;пушистая&raquo;. Собственно, **нужно построить график зависимости размера восстановленной карты от размера кэша в одометрии (`max_lidar_clouds_buffer_size`)**.

In [ ]:
min_distance_between_clouds_cm = 50
max_lidar_clouds_buffer_sizes = list(range(1, 21))
map_voxel_size = 0.25  # Не меняйте этот параметр
frames_range = None

# Baseline map size
restored_map_file_path = get_trajectory_txt_file_path(AlgorithmType.BASELINE, None, None)
assert os.path.isfile(restored_map_file_path)
baseline_restored_map_size = get_restored_map_size(
    read_transforms_matrices_from_file(restored_map_file_path),
    map_voxel_size,
    frames_range=frames_range)
print(f"Restored map size for baseline: {baseline_restored_map_size}")

# Lidar odometry map sizes
restored_map_sizes = []
for max_lidar_clouds_buffer_size in max_lidar_clouds_buffer_sizes:
    # Вот тут вычисляем размер карты
    restored_map_file_path = get_trajectory_txt_file_path(
        AlgorithmType.LIDAR_ODOMETRY,
        max_lidar_clouds_buffer_size,
        min_distance_between_clouds_cm
    )
    if not os.path.isfile(restored_map_file_path):
        print(f"File not found: <{restored_map_file_path}>")
        restored_map_sizes.append(None)
        continue

    restored_lidar_trajectory = read_transforms_matrices_from_file(restored_map_file_path)
    restored_map_size = get_restored_map_size(restored_lidar_trajectory, map_voxel_size, frames_range=frames_range)
    restored_map_sizes.append(restored_map_size)
    print(f"Restored map size for {max_lidar_clouds_buffer_size}: {restored_map_size}")
    del max_lidar_clouds_buffer_size, restored_map_size

del map_voxel_size

In [ ]:
plt.figure(figsize=(10, 8))
plt.plot(
    max_lidar_clouds_buffer_sizes, restored_map_sizes,
    linestyle='-', marker='o', color='r', alpha=0.7)
plt.xticks(ticks=max_lidar_clouds_buffer_sizes)
# plt.hlines(baseline_restored_map_size, 1, 20, label="baseline", color='b')
plt.grid(which='both', linestyle='--', alpha=0.5);
plt.xlabel('max_lidar_clouds_buffer_size')
plt.ylabel('map_size')
plt.legend()
plt.title(f'min_distance_between_clouds_cm={min_distance_between_clouds_cm}');
plt.savefig(f'{DRIVE_RESULTS_DIR_PATH}/restored_map_sizes_for_{min_distance_between_clouds_cm}_cm.png', format='png', dpi=300)

#### Что должно получиться?<sup>[toc](#_toc)</sup>

При используемых выше параметрах одометрии отличие размера карты от бейзлайна будет в разы. Такая картина наблюадется практически на всех проездах KITTY: даже при использовании буфера из одного облака размер итоговой карты обычно получается в 2-3 раза меньше, чем при baseline-подходе, когда восстанавливаем карту просто по показаниям локализации из проезда.